# Volume Cropper: crop boxes out of an OME-Zarr image

This notebook cuts one or more boxes out of a large OME-Zarr image, version 0.6,
0.5 or 0.4, also zipped (.ozx, .zip; for example one made by eubi-bridge), and saves each box as a new
OME-Zarr **in the same format**:
- the same OME-Zarr version, chunk and shard sizes and compression;
- the same pyramid levels, cut out rather than recomputed;
- the OME metadata, plus a record of where the crop came from.

**When to use it:** for big crops, run it on the JupyterHub (or BARD), close to
the data. You can define the boxes in the Volume Cropper GUI on your PC (`volume_cropper.bat`,
then *Save boxes…*) or type them below.

**Kernel:** `volume_cropper`, the environment made from `env/environment.yml`
in the tool folder (register it as a kernel once; see the user guide, *Large
crops*). It needs zarr ≥ 3, numpy, scikit-image, matplotlib and tqdm.
**Your own copy:** copy this notebook into your own folder before you use it,
so that your parameters don't change the shared copy, and set `TOOL_DIR` in
the first code cell to the tool folder.
**Paths:** use the paths as the computer running the notebook sees them: a
network drive has one path on Windows (for example a drive letter) and
another on Linux (the JupyterHub, BARD, the cluster).

Steps: 1 parameters → 2 open the image → 3 plan the boxes → 4 preview →
5 crop and verify → 6 look at the result. The method is explained at the top
of `volume_cropper/core.py`, in the package folder.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

# Where the tool's code (the volume_cropper package) lives. Jupyter starts in
# the notebook's folder, so this notebook, in the tool's examples folder,
# finds the tool one level up by itself. A copy of this notebook kept
# elsewhere (your own copy): set TOOL_DIR to the tool folder, the folder
# with volume_cropper.bat (on Windows, write / instead of backslashes).
TOOL_DIR = None     # e.g. "<tool folder>"; None = the folder above this one
if TOOL_DIR is None:
    for folder in (".", ".."):
        if Path(folder, "volume_cropper", "__init__.py").exists():
            TOOL_DIR = os.path.abspath(folder)
            break
if TOOL_DIR is None or not Path(TOOL_DIR, "volume_cropper",
                                "__init__.py").exists():
    raise FileNotFoundError(
        "Volume Cropper's code was not found. Set TOOL_DIR above to the "
        "tool folder (the folder with volume_cropper.bat and the "
        "volume_cropper folder).")
TOOL_DIR = Path(TOOL_DIR)
sys.path.insert(0, str(TOOL_DIR))
from volume_cropper import core as vc

print("volume_cropper", vc.__version__, "from", TOOL_DIR)

## 1. Parameters

- **Boxes** are in physical units: µm for z, y and x, and the time unit for t.
  An axis left out of a box is kept whole.
- **Channels:** `channels=None` keeps all channels; `channels=[0, 2]` keeps
  channels 0 and 2.
- **Outlines:** boxes saved from the GUI may have outlines: polygons on
  some slices of one or more views. Together they make one smooth 3D
  shape through all of them, which ends at the outermost ones and can be
  grown or shrunk by the box's outline margin (in voxels); polygons drawn
  on a projection are silhouettes through the projected depth. Voxels
  inside it are kept, the others are written as 0. The preview below
  shows the boxes, not the outlines.
- **Enlargement:** each box is enlarged by at most one voxel of the coarsest
  pyramid level kept, so that all levels line up. Step 3 shows the exact
  result.
- **TIFF for Fiji:** `TIFF_LEVEL = 0` (or 1, 2, …) also saves that level of
  each crop as an ImageJ TIFF next to it, `<source name>_<box name>_level<k>.tif`,
  with the voxel size, frame interval, channel names and colours. A crop
  with fewer levels gives its coarsest. TIFFs are not compressed: step 3
  shows their size. `COMPRESS = True` compresses them with zstd (lossless,
  usually about half the size; Fiji then opens them through Bio-Formats).
- **Dry run first:** start with `DRY_RUN = True`. Steps 3–4 then show what
  would be written, and nothing is written until you set it to `False`.

In [ ]:
# Input image (read-only; it is never modified), e.g.
# "<your folder>/<image>.zarr"
SOURCE = None

# Boxes: a CSV saved from the napari GUI ...
BOXES_CSV = None     # e.g. "<your folder>/<image>_boxes.csv"
# ... or typed here (used only when BOXES_CSV is None); replace this
# example with your own ranges in µm.
BOXES = [
    vc.CropBox("crop01", {"z": (100, 200), "y": (100, 300), "x": (100, 300)}),
]
BOX_NAMES = None     # crop only these boxes, e.g. ["crop01"]; None = all

# Output folder: one <source name>_<box name>.zarr per box. Keep crops apart
# from the raw data, e.g. in a "crops" folder of your project.
OUTPUT_DIR = None    # e.g. "<your folder>/crops"

KEEP_POSITION = True   # the crop keeps its place (overlays the full image)
CHANNEL_LABELS = None  # new names for the kept channels, e.g. ["nuclei", "membrane"]
MIN_LEVEL_SIZE = 64    # keep pyramid levels while the crop is >= this many voxels along its longest axis
WORKERS = 8            # blocks copied in parallel; not more than your session's CPUs
OVERWRITE = False      # replace earlier crops (and TIFFs) with the same name
TIFF_LEVEL = None      # also save this level of each crop as a TIFF for Fiji (0 = full resolution); None = no TIFF
COMPRESS = False       # compress those TIFFs with zstd (lossless; Fiji opens them through Bio-Formats)
DRY_RUN = True         # True: plan and preview only; False: write the crops

PREVIEW_CHANNEL = 0    # source channel shown in the previews

## 2. Open the image

Only the metadata are read here, like opening a virtual stack in Fiji. The
list shows the OME-Zarr version, and each pyramid level with its voxel size
and its chunk and shard shape (0.4 images have no shards).

In [ ]:
if SOURCE is None:
    raise ValueError("Set SOURCE (your image) in the parameters cell first.")
from volume_cropper import file_images
if file_images.is_file_image(SOURCE):
    # (0.12.0) TIFF files are read into memory and cropped into TIFFs:
    # done in the window or on the command line, not here.
    raise ValueError(
        "This notebook crops OME-Zarr images. For a TIFF file use the "
        "window (volume_cropper.bat) or the command line: python "
        "<tool folder>/volume_cropper/core.py IMAGE.tif BOXES.csv OUTPUT_DIR")
image = vc.open_image(SOURCE)
print(image.describe())

## 3. Plan the boxes

For every box this shows:
- the range requested and the range that will be written;
- which pyramid levels are kept;
- the size of level 0 (uncompressed; on disk it is usually 2–3× smaller).

In [ ]:
boxes = vc.load_boxes_csv(BOXES_CSV) if BOXES_CSV else list(BOXES)
if BOX_NAMES is not None:
    missing = set(BOX_NAMES) - {box.name for box in boxes}
    if missing:
        raise ValueError(f"No box called {sorted(missing)}")
    boxes = [box for box in boxes if box.name in BOX_NAMES]

plans = [vc.plan_crop(image, box, min_size=MIN_LEVEL_SIZE) for box in boxes]
dtype = image.levels[0].array.dtype
for plan in plans:
    print(plan.describe(image))
    if TIFF_LEVEL is not None:
        level = vc.tiff_level_for(plan, TIFF_LEVEL)
        print(f"  TIFF: level {level}, "
              f"{vc.format_bytes(vc.tiff_bytes(plan, dtype, TIFF_LEVEL))}")
    print()

total = sum(vc.nbytes(plan.shapes[0], dtype) for plan in plans)
print(f"{len(plans)} box(es), {vc.format_bytes(total)} uncompressed at level 0")

## 4. Preview

Maximum-intensity projections (MIPs) of a coarse pyramid level, along z (XY
view), y (XZ view) and x (YZ view). The boxes are drawn as they will be
cropped. Each view is like Fiji's *Image › Stacks › Z Project… (Max
Intensity)*, projected along a different axis.

In [ ]:
def mip_views(img, channel=0, max_size=512, t_index=0):
    # Max projections of a 3D (z, y, x) image, from the finest pyramid
    # level whose longest side is <= max_size (small and quick to read).
    # Returns {"xy": (mip, extent), "xz": ..., "yz": ...}, where extent is
    # (left, right, bottom, top) in physical units, as imshow expects.
    spatial = [i for i, a in enumerate(img.axes) if a.get("type") == "space"]
    level = img.levels[-1]
    for candidate in img.levels:
        if max(candidate.array.shape[i] for i in spatial) <= max_size:
            level = candidate
            break
    selection = []          # one channel and timepoint, all of z, y, x
    for axis in img.axes:
        if axis.get("type") == "channel":
            selection.append(channel)
        elif axis.get("type") == "time":
            selection.append(t_index)
        else:
            selection.append(slice(None))
    zyx = np.asarray(level.array[tuple(selection)])
    (z0, z1), (y0, y1), (x0, x1) = [
        vc.index_range_to_physical(0, level.array.shape[i], level.scale[i],
                                   level.translation[i])
        for i in spatial]
    # top < bottom, so y and z increase downwards, as in Fiji and napari
    return {"xy": (zyx.max(axis=0), (x0, x1, y1, y0)),
            "xz": (zyx.max(axis=1), (x0, x1, z1, z0)),
            "yz": (zyx.max(axis=2), (y0, y1, z1, z0))}


def show_views(views, plans=(), title=""):
    # Show the three MIPs side by side, with the crop boxes drawn on top.
    axis_names = {"xy": ("x", "y"), "xz": ("x", "z"), "yz": ("y", "z")}
    projected = {"xy": "z", "xz": "y", "yz": "x"}   # axis of the max
    fig, axes = plt.subplots(1, 3, figsize=(16, 6), layout="constrained")
    for ax, (name, (mip, extent)) in zip(axes, views.items()):
        low, high = np.percentile(mip, [0.5, 99.8])   # auto-contrast
        ax.imshow(mip, extent=extent, cmap="gray", vmin=low, vmax=high,
                  interpolation="nearest")
        h_axis, v_axis = axis_names[name]
        for k, plan in enumerate(plans):
            (h0, h1) = plan.snapped_ranges[h_axis]
            (v0, v1) = plan.snapped_ranges[v_axis]
            ax.add_patch(plt.Rectangle((h0, v0), h1 - h0, v1 - v0, fill=False,
                                       edgecolor=f"C{k}", linewidth=1.5))
            ax.text(h0, v0, plan.box.name, color=f"C{k}", va="bottom")
        ax.set_title(f"{name.upper()} (max along {projected[name]})")
        ax.set_xlabel(f"{h_axis} (µm)")
        ax.set_ylabel(f"{v_axis} (µm)")
    fig.suptitle(title)
    plt.show()


show_views(mip_views(image, channel=PREVIEW_CHANNEL), plans,
           title=f"{image.path.name}, channel {PREVIEW_CHANNEL}")

## 5. Crop and verify

- **Output name:** each box is written to
  `OUTPUT_DIR/<source name>_<box name>.zarr`, in the image's OME-Zarr version.
- **Copying:** data are copied one block at a time, `WORKERS` blocks in
  parallel: a block is a shard (0.5), or about 32 MB of whole chunks (0.4,
  which has no shards).
- **Interrupted crops:** the OME metadata are written last, so an interrupted
  crop has no OME metadata. Run it again with `OVERWRITE = True` to replace
  it.
- **Check:** after each crop, `verify_crop` compares it with the source: the
  position and voxel size of every level (recomputed from the plan), the
  channel names, the format (version, shape, chunks, shards, compression), and
  the voxels of a few sample blocks per level (a spot check, not every voxel).
  The result is stored in the crop's record (`verified` in `crop_provenance`).
- **TIFF** (with `TIFF_LEVEL`): after the crop, `export_tiff` saves the level
  next to it, a few slices at a time, and `verify_tiff` checks it against the
  crop (size, voxel size and the voxels of a few slices). In Fiji open it
  with *File › Open* (a TIFF over 4 GB uses ImageJ's own large-file layout,
  which other programs may not read fully).

In [ ]:
results = []
if DRY_RUN:
    print("DRY_RUN is True, so nothing was written. Check steps 3-4, then set "
          "DRY_RUN = False and run the notebook again.")
else:
    if OUTPUT_DIR is None:
        raise ValueError("Set OUTPUT_DIR in the parameters cell first.")
    out_dir = Path(OUTPUT_DIR)
    # Check every box before the first is written: never inside the
    # source, no two boxes saved under one name, no foreign folders
    # (or TIFFs).
    vc.check_output_folder(image, out_dir)
    vc.check_box_names([plan.box for plan in plans])
    if TIFF_LEVEL is not None:
        vc.check_tiff_image(image)       # data type and axes fit a TIFF
    for plan in plans:
        out_path = vc.output_path_for(image, plan.box, out_dir)
        vc.check_output(image, plan.box, out_path, OVERWRITE)
        if TIFF_LEVEL is not None:
            vc.check_tiff_output(
                vc.tiff_path_for(out_path, vc.tiff_level_for(plan, TIFF_LEVEL)),
                OVERWRITE, plan.box.name)
    out_dir.mkdir(parents=True, exist_ok=True)
    for plan in plans:
        out_path = vc.output_path_for(image, plan.box, out_dir)
        if OVERWRITE:   # TIFFs of the crop being replaced would show the old region
            for path in vc.remove_old_tiffs(out_path, plan.box.name):
                print(f"deleted the old TIFF {path.name}")
        with tqdm(desc=plan.box.name, unit="block") as bar:
            def progress(done, total):
                bar.total = total
                bar.update(done - bar.n)

            t0 = time.perf_counter()
            vc.write_crop(image, plan, out_path, keep_position=KEEP_POSITION,
                          channel_labels=CHANNEL_LABELS, workers=WORKERS,
                          overwrite=OVERWRITE, progress=progress)
            seconds = time.perf_counter() - t0
        problems = vc.verify_crop(image, plan, out_path,
                                  keep_position=KEEP_POSITION,
                                  channel_labels=CHANNEL_LABELS)
        results.append(vc.CropResult(out_path, plan, seconds, problems))
        print(f"{out_path}\n  {seconds:.0f} s; "
              + vc.verification_text(plan, problems))
        if TIFF_LEVEL is not None:
            level = vc.tiff_level_for(plan, TIFF_LEVEL)
            with tqdm(desc=f"{plan.box.name} TIFF", unit="B", unit_scale=True) as bar:
                def tiff_progress(done, total):
                    bar.total = total
                    bar.update(done - bar.n)

                t0 = time.perf_counter()
                tiff = vc.export_tiff(out_path, level, overwrite=OVERWRITE,
                                      compression="zstd" if COMPRESS else None,
                                      progress_bytes=tiff_progress)
                seconds = time.perf_counter() - t0
            tiff_problems = vc.verify_tiff(out_path, tiff, level)
            print(f"{tiff}\n  level {level}, {seconds:.0f} s; "
                  + ("checked OK" if not tiff_problems
                     else "CHECK FAILED: " + "; ".join(tiff_problems)))

## 6. Look at the result

These are XY/XZ/YZ projections of each crop. With `KEEP_POSITION = True` the
axes show the same µm coordinates as the full image. To browse a crop in 3D,
open it in the napari GUI (*Open…*).

In [ ]:
for result in results:
    crop = vc.open_image(result.path)
    kept = result.plan.channels or [0]
    channel = kept.index(PREVIEW_CHANNEL) if PREVIEW_CHANNEL in kept else 0
    show_views(mip_views(crop, channel=channel),
               title=f"{result.path.name}, channel {kept[channel]} of the source")

## Notes

- **Provenance:** each crop records where it came from, under
  `crop_provenance` in its `zarr.json`: source, requested and written box,
  channels, levels kept, date and software versions.
- **Speed:** measured on 2026-10-21 on a Windows PC, reading over the
  network drive: about 100 MB/s (uncompressed), so a 300 µm cube with 3 channels
  (1.3 GB) took about 12 s. On 2026-11-22 a large crop ran at about
  120 MB/s in this notebook on the JupyterHub (8 workers), and at about
  300 MB/s as a cluster job (16 workers).
- **Very large crops** (hundreds of GB) can run unattended as a cluster job,
  using the command-line form:
  `python <tool folder>/volume_cropper/core.py SOURCE boxes.csv OUTPUT_DIR`
  (add `--dry-run` to see the plan only).